# Price Intelligence — Pipeline Statistics Validation
**Role: Data Engineer**  
**Purpose: Validate data quality and distributions from dbt pipeline output**

Pipeline: `Scrapy → NiFi (streaming) → Bigtable → Airflow (batch) → dbt (staging → cleaned → aggregated) → BigQuery`

Libraries: `SciPy` · `statsmodels` · `pingouin` · `pandas` · `numpy`

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics libraries (required by project spec)
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded: scipy, statsmodels, pingouin')

## 1. Load Data — dbt Pipeline Output

In [ ]:
# ── Load from BigQuery (dbt output) or local fallback ────────────────────────
import os

LOCAL_PATH = '../data/processed/all_products.json'

try:
    from google.cloud import bigquery
    PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'price_intelligence')
    DATASET    = 'price_intelligence_dbt'
    client     = bigquery.Client(project=PROJECT_ID)
    query = f"""
        SELECT *
        FROM `{PROJECT_ID}.{DATASET}.prices_cleaned`
        WHERE price > 0
    """
    df = client.query(query).to_dataframe()
    print(f'✅ Loaded from BigQuery: {len(df):,} records')
except Exception as e:
    print(f'⚠️  BigQuery unavailable ({e})')
    print('📂 Loading from local pipeline output...')
    df = pd.read_json(LOCAL_PATH)
    df = df[df['price'] > 0].copy()
    print(f'✅ Loaded from local: {len(df):,} records')

print(f'Columns: {list(df.columns)}')
df.head()

## 2. Descriptive Statistics — Pipeline Data Quality Check

In [ ]:
# ── Descriptive Stats ────────────────────────────────────────────────────────
print('=== PRICE DISTRIBUTION ===')
print(df['price'].describe().round(2))

print('\n=== DISCOUNT DISTRIBUTION ===')
if 'discount_pct' in df.columns:
    print(df['discount_pct'].describe().round(2))

print('\n=== NULL CHECK (Data Quality) ===')
print(df.isnull().sum())

print('\n=== RECORDS PER SOURCE ===')
print(df['source'].value_counts())

print('\n=== RECORDS PER CATEGORY ===')
print(df['category'].value_counts())

In [ ]:
# ── Distribution Metrics ─────────────────────────────────────────────────────
print('=== PIPELINE OUTPUT METRICS ===')
print(f"Mean price     : {df['price'].mean():.2f} MAD")
print(f"Median price   : {df['price'].median():.2f} MAD")
print(f"Std deviation  : {df['price'].std():.2f}")
print(f"Skewness       : {df['price'].skew():.4f}")
print(f"Kurtosis       : {df['price'].kurtosis():.4f}")
print(f"Min price      : {df['price'].min():.2f} MAD")
print(f"Max price      : {df['price'].max():.2f} MAD")

In [ ]:
# ── Visualization: Price Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price'].clip(upper=df['price'].quantile(0.95)), 
             bins=50, color='#58a6ff', edgecolor='white', alpha=0.8)
axes[0].set_title('Price Distribution (dbt pipeline output)')
axes[0].set_xlabel('Price (MAD)')
axes[0].set_ylabel('Count')

# Boxplot by source
sources = df['source'].unique()
data_by_source = [df[df['source'] == s]['price'].clip(upper=5000) for s in sources]
axes[1].boxplot(data_by_source, labels=sources)
axes[1].set_title('Price Distribution by Source')
axes[1].set_xlabel('Source')
axes[1].set_ylabel('Price (MAD)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

## 3. Inferential Statistics — SciPy

In [ ]:
# ── T-Test (SciPy): Price differences between sources ────────────────────────
sources = df['source'].unique()
print(f'Sources in pipeline: {sources}')

if len(sources) >= 2:
    group_a = df[df['source'] == sources[0]]['price'].dropna()
    group_b = df[df['source'] == sources[1]]['price'].dropna()

    t_stat, p_value = stats.ttest_ind(group_a, group_b)
    print(f'\n=== T-TEST (SciPy): {sources[0]} vs {sources[1]} ===')
    print(f'T-statistic : {t_stat:.4f}')
    print(f'P-value     : {p_value:.6f}')
    ci = stats.t.interval(0.95, df=len(group_a)-1, 
                           loc=group_a.mean(), 
                           scale=stats.sem(group_a))
    print(f'95% CI ({sources[0]}): [{ci[0]:.2f}, {ci[1]:.2f}]')
    if p_value < 0.05:
        print('✅ Result: Significant price difference between sources')
    else:
        print('Result: No significant price difference between sources')

In [ ]:
# ── ANOVA (SciPy): Price variation across categories ─────────────────────────
categories = df['category'].dropna().unique()
groups = [df[df['category'] == cat]['price'].dropna() for cat in categories]
groups = [g for g in groups if len(g) > 1]

f_stat, p_value = stats.f_oneway(*groups)
print('=== ANOVA (SciPy): Price variation across categories ===')
print(f'F-statistic : {f_stat:.4f}')
print(f'P-value     : {p_value:.6f}')
if p_value < 0.05:
    print('✅ Result: Significant price variation across categories')
else:
    print('Result: No significant price variation across categories')

In [ ]:
# ── Mann-Whitney U Test (SciPy) ───────────────────────────────────────────────
if len(sources) >= 2:
    group_a = df[df['source'] == sources[0]]['price'].dropna()
    group_b = df[df['source'] == sources[1]]['price'].dropna()
    u_stat, p_value = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
    print(f'=== Mann-Whitney U Test: {sources[0]} vs {sources[1]} ===')
    print(f'U-statistic : {u_stat:.4f}')
    print(f'P-value     : {p_value:.6f}')
    if p_value < 0.05:
        print('✅ Result: Significant difference (non-parametric)')

## 4. Regression Analysis — statsmodels

In [ ]:
# ── OLS Regression (statsmodels): Price ~ discount_pct + rating ──────────────
cols = ['price', 'discount_pct']
if 'rating' in df.columns:
    cols.append('rating')

reg_df = df[cols].dropna()

formula = 'price ~ discount_pct'
if 'rating' in reg_df.columns:
    formula += ' + rating'

model = smf.ols(formula=formula, data=reg_df).fit()
print('=== OLS REGRESSION (statsmodels) ===')
print(model.summary())

## 5. Advanced Stats — pingouin

In [ ]:
# ── T-Test with pingouin (effect size + power) ───────────────────────────────
if len(sources) >= 2:
    pg_df = df[df['source'].isin([sources[0], sources[1]])][['price', 'source']].dropna()
    result = pg.ttest(
        pg_df[pg_df['source'] == sources[0]]['price'],
        pg_df[pg_df['source'] == sources[1]]['price']
    )
    print(f'=== T-TEST (pingouin): {sources[0]} vs {sources[1]} ===')
    print(result.to_string())
    print('\n→ pingouin gives: T, dof, p-value, CI95%, Cohen-d, power')

In [ ]:
# ── ANOVA with pingouin ───────────────────────────────────────────────────────
anova_df = df[['price', 'category']].dropna()
# Keep categories with at least 5 records
valid_cats = anova_df['category'].value_counts()
valid_cats = valid_cats[valid_cats >= 5].index
anova_df = anova_df[anova_df['category'].isin(valid_cats)]

result = pg.anova(data=anova_df, dv='price', between='category')
print('=== ANOVA (pingouin): Price across categories ===')
print(result.to_string())
print('\n→ pingouin gives: F, p-value, eta-squared (effect size)')

In [ ]:
# ── Correlation (pingouin) ────────────────────────────────────────────────────
if 'discount_pct' in df.columns:
    corr_df = df[['price', 'discount_pct']].dropna()
    result = pg.corr(corr_df['price'], corr_df['discount_pct'])
    print('=== CORRELATION: Price vs Discount (pingouin) ===')
    print(result.to_string())

## 6. Pipeline Summary Report

In [ ]:
# ── Final Pipeline Validation Summary ────────────────────────────────────────
total     = len(df)
nulls     = df.isnull().sum().sum()
sources_n = df['source'].nunique()
cats_n    = df['category'].nunique() if 'category' in df.columns else 'N/A'

print('=========================================')
print('   PIPELINE VALIDATION SUMMARY')
print('=========================================')
print(f'Total records processed : {total:,}')
print(f'Null values remaining   : {nulls}')
print(f'Sources                 : {sources_n}')
print(f'Categories              : {cats_n}')
print(f'Avg price               : {df["price"].mean():.2f} MAD')
if 'discount_pct' in df.columns:
    print(f'Avg discount            : {df["discount_pct"].mean():.2f}%')
print('=========================================')
print('Libraries used: SciPy · statsmodels · pingouin')
print('Pipeline: Scrapy → NiFi → Bigtable → Airflow → dbt → BigQuery ✅')